# 02 Campaign Propensity

**SYNTHETIC DEMO MODEL**. Reusable production logic lives in `ml/src`. This notebook inspects completed immutable evidence and never tunes against the test set.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_ml_pipeline.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from ml.src.utils.config import config, path
import pandas as pd
from IPython.display import display, Image
cfg = config()
reports = path(cfg, 'reports')

## Business problem and target

Rank marketing audiences. Conversion within 30 days of actual campaign launch. Previously contacted campaign/customer interactions; features frozen at month start. Horizon: 30 days. Feature cutoff is before all label events. These metrics do not establish real-bank usefulness.

In [ ]:
entry = json.loads((reports / 'campaign_latest.json').read_text())
folder = Path(entry['report_path'])
display(pd.DataFrame(entry['periods']).T)
training = pd.read_parquet(path(cfg, 'features') / 'campaign_training.parquet')
display(training.groupby('prediction_date').target.agg(['count', 'sum', 'mean']))

## Baselines, trees, temporal CV and model comparison

Logistic regression, random forest and histogram gradient boosting compete with a constant prior. Class weights are training-only. The configured grids are selected with purged expanding-window CV. The champion is chosen using later-validation ranking/stability before test evaluation.

In [ ]:
display(pd.read_csv(folder / 'model_comparison.csv'))
display(pd.read_csv(folder / 'cross_validation.csv'))
display(pd.DataFrame([entry['metrics']]))

## Lift, capacity and calibration

The following held-out tables are descriptive. Do not choose a new threshold or model from them. Calibration fits an earlier validation period only.

In [ ]:
display(pd.read_csv(folder / 'deciles.csv'))
display(pd.read_csv(folder / 'capacity.csv'))
display(entry['calibration_comparison'])
display(Image(filename=str(folder / 'evaluation.png')))

## Explainability and business interpretation

Permutation importance is validation-based and associational. A constant model has no personalized drivers. Inactivity is not verified churn. Weak ranking must remain visible.

In [ ]:
display(pd.read_csv(folder / 'feature_importance.csv').head(12))
display(pd.read_csv(folder / 'local_explanation.csv'))
display(pd.read_csv(folder / 'fairness_audit.csv'))

In [ ]:
display(pd.read_csv(folder / 'targeting_efficiency_simulation.csv'))

## Reproduce and limitations

Run the documented CLI to create a new experiment/version. Notebook execution only reads completed evidence. Models use synthetic behavior, weak holdout results do not justify real targeting, and effect measurement requires randomized treatment/control. See `docs/ml_leakage_prevention.md`, `docs/ml_responsible_ai.md` and the model cards.